In [1]:
from pathlib import Path
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

In [2]:
device = "mps" if torch.backends.mps.is_available() else "cpu"
batch_size = 64
epochs = 10
lr = 1e-3
weights_dir = Path("../model_weights")
weights_dir.mkdir(parents=True, exist_ok=True)

In [3]:
transform = transforms.ToTensor()

train_data = datasets.MNIST(root="../data", train=True, download=True, transform=transform)
test_data = datasets.MNIST(root="../data", train=False, download=True, transform=transform)

train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_data, batch_size=batch_size)

100.0%
100.0%
100.0%
100.0%


In [4]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Flatten(),
            nn.Linear(32 * 7 * 7, 128),
            nn.ReLU(),
            nn.Linear(128, 10),
        )

    def forward(self, x):
        return self.net(x)

model = CNN().to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=lr)

In [5]:
for epoch in range(epochs):
    model.train()
    total_loss = 0

    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        pred = model(x)
        loss = loss_fn(pred, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(device), y.to(device)
            pred = model(x).argmax(dim=1)
            correct += (pred == y).sum().item()
            total += y.size(0)

    print(f"epoch {epoch + 1}/{epochs} loss={total_loss / len(train_loader):.4f} acc={correct / total:.4f}")

epoch 1/10 loss=0.2135 acc=0.9797
epoch 2/10 loss=0.0616 acc=0.9878
epoch 3/10 loss=0.0420 acc=0.9868
epoch 4/10 loss=0.0330 acc=0.9881
epoch 5/10 loss=0.0254 acc=0.9876
epoch 6/10 loss=0.0213 acc=0.9899
epoch 7/10 loss=0.0175 acc=0.9872
epoch 8/10 loss=0.0146 acc=0.9896
epoch 9/10 loss=0.0113 acc=0.9899
epoch 10/10 loss=0.0106 acc=0.9894


In [6]:
torch.save(model.state_dict(), weights_dir / "mnist_cnn.pth")